# research-note_2

## 1 Perceptron

$$
\mathrm{output} =
\begin{cases}
0 & \text{if } w \cdot x + b \le 0 \\
1 & \text{if } w \cdot x + b > 0
\end{cases},
\qquad b \equiv -\text{threshold}
$$

In [1]:
import torch

X = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32)
w = torch.tensor([-2, -2], dtype=torch.float32)
b = torch.tensor([3.0])

y_hat = ( X @ w + b > 0 ).long() # .float() -> tensor([1., 1., 1., 0.])

print(y_hat)

tensor([1, 1, 1, 0])


## 2 Sigmoid Neuron

$$
\mathrm{output} = \sigma(w \cdot x + b) = \frac{1}{1 + \exp\left(-\sum_j w_j x_j - b\right)}
$$

$$
\Delta \mathrm{output} \approx \sum_j \frac{\partial\, \mathrm{output}}{\partial w_j} \Delta w_j + \frac{\partial\, \mathrm{output}}{\partial b} \Delta b
$$

In [2]:
def sigmoid(z):
    return 1 / (1 + torch.exp(-z))

for c in [1, 10, 100]:
    y_hat = sigmoid(X @ (w * c) + b * c)
    print(y_hat)

tensor([0.9526, 0.7311, 0.7311, 0.2689])
tensor([1.0000e+00, 9.9995e-01, 9.9995e-01, 4.5398e-05])
tensor([1., 1., 1., 0.])


In [3]:
print(torch.exp(torch.tensor(100.0)))
print(torch.finfo(torch.float32).max) #overflow

tensor(inf)
3.4028234663852886e+38


## 3 Network Architecture — 784-15-10

$$
h = \sigma(W_1 x + b_1),
\qquad
\hat{y} = \sigma(W_2 h + b_2)
$$

$$
y = (0, 0, 0, 0, 0, 0, 1, 0, 0, 0)^{\top} \quad \text{(one-hot, digit 6)}
$$

In [12]:
def feedforward(X, params):
    for W, b in params:
        X = torch.sigmoid(X @ W + b)
    return X

B = 5
X = torch.randn(B, 784)
W1 = torch.randn(15, 784).T
b1 = torch.randn(15)
W2 = torch.randn(10, 15).T
b2 = torch.randn(10)    
params = [(W1, b1), (W2, b2)]

y_hat = feedforward(X, params)
print(y_hat.shape)
print(y_hat)


torch.Size([5, 10])
tensor([[1.8925e-01, 9.3046e-01, 8.2140e-01, 1.9270e-01, 9.9470e-02, 9.9959e-01,
         6.5703e-01, 4.1794e-02, 9.4027e-01, 4.1755e-01],
        [6.0233e-01, 9.2393e-01, 6.4716e-02, 3.1529e-01, 6.1609e-01, 1.7807e-01,
         8.7011e-01, 4.3438e-03, 7.0847e-01, 8.9947e-01],
        [2.6603e-02, 9.9544e-01, 3.1215e-01, 5.6862e-02, 6.3047e-01, 3.5649e-01,
         6.9738e-01, 2.7349e-03, 9.8354e-01, 5.8930e-01],
        [3.2076e-01, 9.3847e-01, 3.6401e-01, 1.3308e-01, 8.6066e-01, 8.6184e-01,
         7.2813e-01, 8.5792e-02, 9.6949e-01, 5.0734e-01],
        [5.6260e-03, 9.9782e-01, 1.2395e-01, 2.4492e-01, 7.6188e-01, 7.5224e-01,
         9.8482e-01, 2.1553e-04, 9.8484e-01, 1.3410e-01]])


## 4 Quadratic Cost (MSE)

$$
L(w, b) = \frac{1}{2n} \sum_{i=1}^{n} \left\| y^{(i)} - \hat{y}^{(i)} \right\|^2
$$

In [ ]:
labels = torch.tensor([0, 2])
num_classes = 3
y = (labels[:, None] == torch.arange(num_classes)).float()
y_hat = torch.tensor([[0.8, 0.1, 0.1], [0.2, 0.2, 0.6]])
squared_error = torch.sum((y_hat - y) ** 2)
n = len(labels)
L = squared_error / (2 * n)
print(L)


tensor(0.0750)


In [18]:
import torch.nn as nn

mse = nn.MSELoss()
L_mse = mse(y_hat, y)

print(L)        #/4
print(L_mse)    #/6

tensor(0.0750)
tensor(0.0500)


## 5 Gradient Descent

$$
\Delta L \approx \nabla L \cdot \Delta v,
\qquad
\Delta v = -\eta \nabla L
\;\Rightarrow\;
\Delta L \approx -\eta \|\nabla L\|^2 \le 0
$$

$$
v \leftarrow v - \eta \nabla L
$$

In [21]:
def loss(v):
    L = torch.sum(v ** 2)
    return L
def grad_loss(v):
    grad = 2 * v
    return grad
def gradient_descent_step(v, lr):
    grad = grad_loss(v)
    v_new = v - lr * grad
    return v_new

num_steps = 20

for lr in [0.1, 1.0, 1.1]:
    v = torch.tensor([1.0, 2.0], dtype = torch.float32)
    print(f"\nlr = {lr}")
    for step in range(num_steps):
        v = gradient_descent_step(v, lr)
        L = loss(v)
        print(f"step = {step + 1}, L = {L.item():.6f}")


lr = 0.1
step = 1, L = 3.200000
step = 2, L = 2.048000
step = 3, L = 1.310720
step = 4, L = 0.838861
step = 5, L = 0.536871
step = 6, L = 0.343597
step = 7, L = 0.219902
step = 8, L = 0.140737
step = 9, L = 0.090072
step = 10, L = 0.057646
step = 11, L = 0.036893
step = 12, L = 0.023612
step = 13, L = 0.015112
step = 14, L = 0.009671
step = 15, L = 0.006190
step = 16, L = 0.003961
step = 17, L = 0.002535
step = 18, L = 0.001623
step = 19, L = 0.001038
step = 20, L = 0.000665

lr = 1.0
step = 1, L = 5.000000
step = 2, L = 5.000000
step = 3, L = 5.000000
step = 4, L = 5.000000
step = 5, L = 5.000000
step = 6, L = 5.000000
step = 7, L = 5.000000
step = 8, L = 5.000000
step = 9, L = 5.000000
step = 10, L = 5.000000
step = 11, L = 5.000000
step = 12, L = 5.000000
step = 13, L = 5.000000
step = 14, L = 5.000000
step = 15, L = 5.000000
step = 16, L = 5.000000
step = 17, L = 5.000000
step = 18, L = 5.000000
step = 19, L = 5.000000
step = 20, L = 5.000000

lr = 1.1
step = 1, L = 7.200000
step 

## 6 Stochastic Gradient Descent #note1 -> 4

$$
\nabla L \approx \frac{1}{|\mathcal{B}|} \sum_{i \in \mathcal{B}} \nabla \ell^{(i)}
$$

## 7 Feedforward #3

$$
a' = \sigma(W a + b),
\qquad
a'_j = \sigma\left( \sum_k W_{jk} a_k + b_j \right)
$$